# Preprocesamiento y construcción del dataset de modelado

En la notebook `01_exploracion_datos.ipynb` exploramos los datasets de producción, pozos y fracturación, identificamos su granularidad y detectamos distintos problemas de calidad que dejamos explícitamente para la etapa de limpieza.

En esta notebook continuamos ese trabajo. El objetivo es construir una única tabla integrada (*One Big Table*, OBT) que sirva como entrada para las siguientes etapas de ingeniería de variables y modelado.

La tabla resultante tiene la siguiente granularidad:

> **Una fila por pozo y por mes calendario.**

Cada fila contiene la información disponible al cierre de ese mes y, como variable objetivo, la producción de petróleo del mes siguiente.

**Entradas:** los archivos que exporta la sección 5.5 de la notebook 01 en `data/processed/` (`produccion_limpio.csv`, `pozos_limpio.csv` y `fracturacion_limpio.csv`). Por lo tanto, antes de ejecutar esta notebook es necesario ejecutar la notebook 01. Además, para validar las unidades usamos la serie histórica oficial de producción de `data/raw/`.

**Salida:** `data/processed/dataset_modelado.csv`.

Durante todo el proceso seguimos estos criterios:

- no eliminamos registros de producción: las correcciones se aplican sobre los valores y quedan marcadas;
- no completamos valores faltantes con datos inventados;
- no tratamos como errores los valores extremos ni las producciones iguales a cero, ya que en la exploración vimos que pueden corresponder a situaciones operativas reales;
- cada variable de una fila solo puede contener información disponible al cierre del mes de esa fila.

### Importación de librerías

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

# Indicamos que se visualicen todas las columnas del dataframe
pd.set_option("display.max_columns", None)

### 1. Carga de los datos y validaciones iniciales

#### 1.1. Carga de los datasets procesados

Leemos los datasets que exporta la notebook 01. Si alguno de los archivos no existe, mostramos un mensaje que indica que primero hay que ejecutar esa notebook.

In [2]:
ruta_processed = Path("../data/processed")
ruta_raw = Path("../data/raw")

archivos_entrada = {
    "produccion": ruta_processed / "produccion_limpio.csv",
    "pozos": ruta_processed / "pozos_limpio.csv",
    "fracturacion": ruta_processed / "fracturacion_limpio.csv",
}

for nombre, ruta in archivos_entrada.items():
    if not ruta.exists():
        raise FileNotFoundError(
            f"No se encontró {ruta}. Primero hay que ejecutar la notebook 01_exploracion_datos.ipynb."
        )

produccion = pd.read_csv(archivos_entrada["produccion"], low_memory=False)
pozos = pd.read_csv(archivos_entrada["pozos"], low_memory=False)
fracturacion = pd.read_csv(archivos_entrada["fracturacion"], low_memory=False)

for nombre, dataframe in [("Producción", produccion), ("Pozos", pozos), ("Fracturación", fracturacion)]:
    print(f"{nombre}: {dataframe.shape[0]:,} filas y {dataframe.shape[1]} columnas")

Producción: 421,046 filas y 40 columnas
Pozos: 85,611 filas y 26 columnas
Fracturación: 4,890 filas y 30 columnas


#### 1.2. Recuperación de los tipos de datos

Los archivos CSV no guardan el tipo de dato de cada columna. Por eso, las fechas que convertimos en la sección 4 de la notebook 01 vuelven a leerse como texto, y tenemos que convertirlas de nuevo.

Usamos los mismos formatos que en la notebook 01 y, en lugar de `errors="coerce"`, dejamos que la conversión falle si algún valor no respeta el formato. De esta manera, si los archivos de entrada cambian, nos enteramos en lugar de generar valores faltantes sin darnos cuenta.

In [3]:
produccion["fecha_data"] = pd.to_datetime(produccion["fecha_data"], format="%Y-%m-%d")

fechas_pozos = [
    "adjiv_fecha_inicio_perf",
    "adjiv_fecha_fin_perf",
    "adjiv_fecha_inicio_term",
    "adjiv_fecha_fin_term",
]
for columna in fechas_pozos:
    pozos[columna] = pd.to_datetime(pozos[columna], format="%Y-%m-%d")

fracturacion["fecha_inicio_fractura"] = pd.to_datetime(fracturacion["fecha_inicio_fractura"], format="%Y-%m-%d")
fracturacion["fecha_fin_fractura"] = pd.to_datetime(fracturacion["fecha_fin_fractura"], format="%Y-%m-%d")

print("Tipos de las columnas de fecha:")
display(produccion[["fecha_data"]].dtypes)
display(pozos[fechas_pozos].dtypes)
display(fracturacion[["fecha_inicio_fractura", "fecha_fin_fractura"]].dtypes)

print("Tipos de las columnas booleanas:")
display(produccion[["rectificado", "habilitado"]].dtypes)
display(pozos[["gasplus"]].dtypes)

Tipos de las columnas de fecha:


fecha_data    datetime64[us]
dtype: object

adjiv_fecha_inicio_perf    datetime64[us]
adjiv_fecha_fin_perf       datetime64[us]
adjiv_fecha_inicio_term    datetime64[us]
adjiv_fecha_fin_term       datetime64[us]
dtype: object

fecha_inicio_fractura    datetime64[us]
fecha_fin_fractura       datetime64[us]
dtype: object

Tipos de las columnas booleanas:


rectificado    bool
habilitado     bool
dtype: object

gasplus    bool
dtype: object

Guardamos un resumen de cada dataset tal como fue cargado, para compararlo al final con la tabla resultante (sección 5.2).

In [4]:
def resumen(dataframe, columna_fecha=None):
    fila = {
        "filas": len(dataframe),
        "pozos": dataframe["idpozo"].nunique(),
        "columnas": dataframe.shape[1],
        "desde": "-",
        "hasta": "-",
    }
    if columna_fecha is not None:
        fila["desde"] = dataframe[columna_fecha].min().strftime("%Y-%m")
        fila["hasta"] = dataframe[columna_fecha].max().strftime("%Y-%m")
    return fila


resumen_entrada = {
    "Producción (entrada)": resumen(produccion, "fecha_data"),
    "Pozos (entrada)": resumen(pozos),
    "Fracturación (entrada)": resumen(fracturacion, "fecha_fin_fractura"),
}

pd.DataFrame(resumen_entrada).T

,filas,pozos,columnas,desde,hasta
Producción (entrada),421046,5089,40,2006-01,2026-07
Pozos (entrada),85611,85611,26,-,-
Fracturación (entrada),4890,4646,30,2006-05,2026-07


#### 1.3. Identificadores y llaves

Antes de limpiar o combinar los datos, verificamos las llaves que vamos a usar:

- en producción, la combinación de `idpozo`, `anio` y `mes` debe ser única;
- en pozos, `idpozo` debe ser único;
- en fracturación, cada registro tiene su propio identificador (`id_base_fractura_adjiv`), pero un pozo puede tener varios registros.

También verificamos que no haya identificadores faltantes y que `fecha_data` corresponda al mismo año y mes que `anio` y `mes`, tal como vimos en la notebook 01.

In [5]:
print("idpozo faltantes - producción:", produccion["idpozo"].isna().sum())
print("idpozo faltantes - pozos:", pozos["idpozo"].isna().sum())
print("idpozo faltantes - fracturación:", fracturacion["idpozo"].isna().sum())

print()
print("Duplicados de idpozo, anio y mes en producción:", produccion.duplicated(["idpozo", "anio", "mes"]).sum())
print("Duplicados de idpozo en pozos:", pozos["idpozo"].duplicated().sum())
print("Duplicados de id_base_fractura_adjiv en fracturación:", fracturacion["id_base_fractura_adjiv"].duplicated().sum())

coincide_periodo = (
    (produccion["fecha_data"].dt.year == produccion["anio"])
    & (produccion["fecha_data"].dt.month == produccion["mes"])
)
print()
print("Registros donde fecha_data no coincide con anio y mes:", (~coincide_periodo).sum())

idpozo faltantes - producción: 0
idpozo faltantes - pozos: 0
idpozo faltantes - fracturación: 0

Duplicados de idpozo, anio y mes en producción: 0
Duplicados de idpozo en pozos: 0
Duplicados de id_base_fractura_adjiv en fracturación: 0

Registros donde fecha_data no coincide con anio y mes: 0


#### 1.4. Integridad referencial

La producción va a ser la tabla principal del dataset. Por eso medimos qué parte de los pozos con producción tiene información en los otros dos datasets.

In [6]:
pozos_produccion = set(produccion["idpozo"])
pozos_caracteristicas = set(pozos["idpozo"])
pozos_fracturacion = set(fracturacion["idpozo"])

total_pozos = len(pozos_produccion)
con_caracteristicas = len(pozos_produccion & pozos_caracteristicas)
con_fracturacion = len(pozos_produccion & pozos_fracturacion)

print("Pozos con producción:", total_pozos)
print(f"Con información en el dataset de pozos: {con_caracteristicas} ({con_caracteristicas / total_pozos * 100:.1f}%)")
print(f"Con información de fracturación: {con_fracturacion} ({con_fracturacion / total_pozos * 100:.1f}%)")
print()
print("Pozos de fracturación que no están en producción:", len(pozos_fracturacion - pozos_produccion))

Pozos con producción: 5089
Con información en el dataset de pozos: 5006 (98.4%)
Con información de fracturación: 3835 (75.4%)

Pozos de fracturación que no están en producción: 811


Casi todos los pozos con producción figuran en el dataset de pozos, pero aproximadamente una cuarta parte no tiene información de fracturación.

La ausencia de datos de fracturación no significa que el pozo no haya sido fracturado: puede tratarse de información no cargada. Por eso, al integrar los datos, estos pozos van a tener valores faltantes en las variables de fracturación, y no valores iguales a cero.

Por otro lado, el dataset de fracturación incluye pozos que no están en el dataset de producción no convencional. Como la producción es la tabla principal, esos registros no se incorporan.

#### 1.5. Validación de las unidades de medida

Los datasets no informan explícitamente las unidades de las variables de producción. Para verificarlas, comparamos la suma mensual de `prod_pet` y `prod_gas` con la serie histórica oficial de producción no convencional (shale y tight), que forma parte de los archivos descargados.

La serie de petróleo incluye además la producción diaria en miles de barriles (`kbbl_diario`), lo que nos permite confirmar la unidad: un metro cúbico equivale a 6,2898 barriles.

In [7]:
serie_petroleo = pd.read_csv(
    ruta_raw / "serie-histrica-de-produccin-de-petrleo-por-cuenca-y-sub-tipo-de-recurso-captulo-iv-.csv"
)
serie_gas = pd.read_csv(
    ruta_raw / "serie-histrica-de-produccin-de-gas-natural-por-cuenca-y-sub-tipo-de-recurso-captulo-iv-.csv"
)

serie_petroleo["fecha"] = pd.to_datetime(serie_petroleo["indice_tiempo"]).dt.to_period("M")
serie_gas["fecha"] = pd.to_datetime(serie_gas["indice_tiempo"]).dt.to_period("M")

produccion_mensual = (
    produccion
    .assign(fecha=produccion["fecha_data"].dt.to_period("M"))
    .groupby("fecha")[["prod_pet", "prod_gas"]]
    .sum()
)

comparacion = (
    produccion_mensual
    .join(serie_petroleo.set_index("fecha")[["total", "total_diario", "shale", "tight", "kbbl_diario"]])
    .join(serie_gas.set_index("fecha")[["produccion_shale_gas", "produccion_tight_gas"]])
)

comparacion["ratio_petroleo"] = comparacion["prod_pet"] / (comparacion["shale"] + comparacion["tight"])
comparacion["ratio_gas"] = comparacion["prod_gas"] / (comparacion["produccion_shale_gas"] + comparacion["produccion_tight_gas"])
comparacion["kbbl_diario_calculado"] = comparacion["total_diario"] * 6.2898 / 1000

print("Relación entre la suma de prod_pet y la serie oficial (shale + tight):")
display(comparacion["ratio_petroleo"].describe())

print("Relación entre la suma de prod_gas y la serie oficial (shale + tight):")
display(comparacion["ratio_gas"].describe())

print("Conversión de la producción total diaria a miles de barriles:")
display(comparacion[["total_diario", "kbbl_diario", "kbbl_diario_calculado"]].tail(3))

Relación entre la suma de prod_pet y la serie oficial (shale + tight):


count    247.000000
mean       0.999443
std        0.005695
min        0.922021
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: ratio_petroleo, dtype: float64

Relación entre la suma de prod_gas y la serie oficial (shale + tight):


count    247.000000
mean       0.999759
std        0.002114
min        0.974761
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: ratio_gas, dtype: float64

Conversión de la producción total diaria a miles de barriles:


,total_diario,kbbl_diario,kbbl_diario_calculado
fecha,,,
2026-05,143721.6349,903.9819,903.980339
2026-06,144751.9302,910.4623,910.460691
2026-07,144700.1324,910.1365,910.134893


La suma de `prod_pet` y de `prod_gas` coincide con la serie oficial en prácticamente todos los meses (la relación es igual a 1 en al menos el 75% de los meses). Además, al convertir la producción total diaria de la serie de metros cúbicos a barriles obtenemos exactamente el valor de `kbbl_diario`.

Por lo tanto:

- `prod_pet` está expresada en **metros cúbicos (m³)**;
- `prod_gas` coincide con la serie oficial de gas. Por su magnitud (la suma mensual de 2025 ronda los 2,5 millones), corresponde a **miles de metros cúbicos (Mm³)**: equivale a unos 85 millones de m³ diarios de gas no convencional, un valor razonable para el país. Si la unidad fuera m³, la producción sería mil veces menor.

Las unidades son las mismas en todo el dataset, por lo que no es necesario convertirlas. La serie histórica solo la usamos para esta validación: como es un agregado de todos los pozos, no la incorporamos a la tabla final.

### 2. Limpieza final y consistencia

En esta sección aplicamos el tratamiento de los problemas que identificamos en la sección 5 de la notebook 01.

Antes de modificar cualquier valor, guardamos la cantidad de filas de cada dataset para poder verificar al final que no se eliminaron registros.

In [8]:
filas_iniciales = {
    "produccion": len(produccion),
    "pozos": len(pozos),
    "fracturacion": len(fracturacion),
}
filas_iniciales

{'produccion': 421046, 'pozos': 85611, 'fracturacion': 4890}

#### 2.1. Producción negativa

En la sección 5.1 de la notebook 01 encontramos tres registros con producción negativa. Al revisar los meses anteriores y posteriores concluimos que:

- los dos registros de gas negativo de mayo de 2020 probablemente corresponden a un error de signo, ya que en positivo encajan con la producción de los meses cercanos;
- el registro de petróleo de -0,001 corresponde a un pozo sin actividad en ese mes, por lo que su valor real es 0.

Aplicamos esas correcciones y marcamos los registros en la columna `prod_corregida`.

Antes de corregir, verificamos que no haya valores negativos en otras variables de producción o inyección.

In [9]:
columnas_volumen = ["prod_pet", "prod_gas", "prod_agua", "iny_agua", "iny_gas", "tef"]

print("Valores negativos por columna:")
display((produccion[columnas_volumen] < 0).sum())

gas_negativo = produccion["prod_gas"] < 0
petroleo_negativo = produccion["prod_pet"] < 0

display(produccion.loc[gas_negativo | petroleo_negativo, ["idpozo", "anio", "mes", "prod_pet", "prod_gas", "tef", "tipoestado"]])

Valores negativos por columna:


prod_pet     1
prod_gas     2
prod_agua    0
iny_agua     0
iny_gas      0
tef          0
dtype: int64

,idpozo,anio,mes,prod_pet,prod_gas,tef,tipoestado
300275,153228,2020,5,1.670,-12.267,6.13,Extracción Efectiva
300283,153227,2020,5,1.020,-7.519,6.13,Extracción Efectiva
323445,73804,2008,12,-0.001,0.000,0.00,En Estudio


In [10]:
produccion["prod_corregida"] = gas_negativo | petroleo_negativo

produccion.loc[gas_negativo, "prod_gas"] = -produccion.loc[gas_negativo, "prod_gas"]
produccion.loc[petroleo_negativo, "prod_pet"] = 0

print("Registros corregidos:", produccion["prod_corregida"].sum())
print("Valores negativos restantes:", (produccion[columnas_volumen] < 0).sum().sum())

Registros corregidos: 3
Valores negativos restantes: 0


#### 2.2. Días efectivos de producción (`tef`)

En la sección 5.1 de la notebook 01 encontramos 71 registros con más días efectivos que los que tiene el mes. La mayoría corresponde a valores copiados del mes anterior que se exceden en pocos días, mientras que unos pocos tienen valores que no pueden recuperarse, como 46,88 o 79,34 días.

Aplicamos la propuesta de tratamiento definida en esa sección:

- si el exceso es de hasta 5 días, truncamos el `tef` a la cantidad de días del mes;
- si el exceso es mayor a 5 días, el valor no puede recuperarse y lo reemplazamos por un valor faltante.

El umbral de 5 días separa con claridad los dos grupos: como se ve a continuación, los valores que truncamos se exceden como máximo en 3 días, mientras que los irrecuperables se exceden en más de 6.

En ambos casos marcamos el registro en la columna `tef_corregido`, para dejar registro de que el valor original fue modificado.

In [11]:
dias_del_mes = produccion["fecha_data"].dt.days_in_month
exceso_tef = produccion["tef"] - dias_del_mes

tef_truncable = (exceso_tef > 0) & (exceso_tef <= 5)
tef_irrecuperable = exceso_tef > 5

print("Registros con tef mayor a los días del mes:", (exceso_tef > 0).sum())
print("Registros a truncar (exceso de hasta 5 días):", tef_truncable.sum())
print("Registros irrecuperables (exceso mayor a 5 días):", tef_irrecuperable.sum())

print("Distribución del exceso en los registros a truncar:")
display(exceso_tef[tef_truncable].describe())

print("Registros irrecuperables:")
display(produccion.loc[tef_irrecuperable, ["idpozo", "anio", "mes", "tef", "prod_gas"]])

Registros con tef mayor a los días del mes: 71
Registros a truncar (exceso de hasta 5 días): 66
Registros irrecuperables (exceso mayor a 5 días): 5
Distribución del exceso en los registros a truncar:


count    66.000000
mean      1.054258
std       0.694177
min       0.200000
25%       0.700000
50%       1.000000
75%       1.000000
max       3.000000
dtype: float64

Registros irrecuperables:


,idpozo,anio,mes,tef,prod_gas
281488,154790,2019,5,79.34,79.34
283068,161531,2021,6,36.67,126.55
283073,161532,2021,6,36.67,129.38
366314,145043,2017,2,46.88,46.88
375581,10073,2006,12,58.77,2428.00


In [12]:
produccion["tef_corregido"] = tef_truncable | tef_irrecuperable

produccion.loc[tef_truncable, "tef"] = dias_del_mes[tef_truncable]
produccion.loc[tef_irrecuperable, "tef"] = np.nan

print("Registros con tef corregido:", produccion["tef_corregido"].sum())
print("Registros con tef mayor a los días del mes después de la corrección:", (produccion["tef"] > dias_del_mes).sum())

Registros con tef corregido: 71
Registros con tef mayor a los días del mes después de la corrección: 0


#### 2.3. Profundidad

En las secciones 3.7 y 5.1 de la notebook 01 encontramos dos tipos de profundidades imposibles:

- valores iguales a 0, que corresponden a profundidades no informadas;
- un pozo con 378.939 metros, un valor claramente erróneo.

Como no podemos saber cuál es la profundidad real, reemplazamos estos valores por faltantes. Para identificarlos usamos el mismo criterio de la notebook 01: profundidad igual a 0 o mayor a 10.000 metros.

In [13]:
profundidad_invalida = (produccion["profundidad"] == 0) | (produccion["profundidad"] > 10000)

print("Registros con profundidad inválida:", profundidad_invalida.sum())
print("Pozos con profundidad inválida:", produccion.loc[profundidad_invalida, "idpozo"].nunique())

produccion.loc[profundidad_invalida, "profundidad"] = np.nan

print("Profundidad mínima y máxima después de la corrección:", produccion["profundidad"].min(), "-", produccion["profundidad"].max())

Registros con profundidad inválida: 5860
Pozos con profundidad inválida: 165
Profundidad mínima y máxima después de la corrección: 2.454 - 8376.0


#### 2.4. Coordenadas invertidas

En la sección 5.1 de la notebook 01 encontramos un pozo con la longitud y la latitud intercambiadas. Usamos el mismo criterio para identificarlo (longitud mayor a -50 y latitud menor a -60) e intercambiamos los valores.

Después verificamos que todas las coordenadas queden dentro del rango aproximado de Argentina: longitud entre -74 y -53, y latitud entre -56 y -21.

In [14]:
coordenadas_invertidas = (produccion["coordenadax"] > -50) & (produccion["coordenaday"] < -60)

print("Registros con coordenadas invertidas:", coordenadas_invertidas.sum())
print("Pozos con coordenadas invertidas:", produccion.loc[coordenadas_invertidas, "idpozo"].nunique())

produccion.loc[coordenadas_invertidas, ["coordenadax", "coordenaday"]] = (
    produccion.loc[coordenadas_invertidas, ["coordenaday", "coordenadax"]].values
)

fuera_de_rango = ~(
    produccion["coordenadax"].between(-74, -53)
    & produccion["coordenaday"].between(-56, -21)
)
print("Registros con coordenadas fuera de Argentina después de la corrección:", fuera_de_rango.sum())

Registros con coordenadas invertidas: 68
Pozos con coordenadas invertidas: 1
Registros con coordenadas fuera de Argentina después de la corrección: 0


#### 2.5. Fracturación: registros duplicados

En la notebook 01 no encontramos registros completamente duplicados en fracturación, porque cada registro tiene su propio identificador y su fecha de carga. Sin embargo, al comparar solo el contenido de cada fractura (pozo, formación, fechas y valores operativos) aparecen eventos repetidos: la misma fractura fue cargada dos veces, en algunos casos con meses de diferencia.

Estos duplicados son importantes porque en la sección 3 vamos a sumar la cantidad de fracturas y los volúmenes inyectados de cada pozo. Si los dejamos, esos eventos se contarían dos veces. Por eso los eliminamos y conservamos la primera carga de cada evento.

In [15]:
columnas_evento = [
    "idpozo",
    "formacion_productiva",
    "fecha_inicio_fractura",
    "fecha_fin_fractura",
    "cantidad_fracturas",
    "longitud_rama_horizontal_m",
    "agua_inyectada_m3",
    "arena_bombeada_nacional_tn",
    "arena_bombeada_importada_tn",
    "presion_maxima_psi",
    "potencia_equipos_fractura_hp",
]

eventos_duplicados = fracturacion.duplicated(columnas_evento, keep=False)
display(
    fracturacion.loc[eventos_duplicados, columnas_evento[:4] + ["cantidad_fracturas", "anio_carga", "mes_carga"]]
    .sort_values(["idpozo", "anio_carga", "mes_carga"])
)

fracturacion = (
    fracturacion
    .sort_values(["anio_carga", "mes_carga", "id_base_fractura_adjiv"])
    .drop_duplicates(columnas_evento, keep="first")
    .reset_index(drop=True)
)

print("Registros eliminados:", filas_iniciales["fracturacion"] - len(fracturacion))
print("Registros de fracturación restantes:", len(fracturacion))

,idpozo,formacion_productiva,fecha_inicio_fractura,fecha_fin_fractura,cantidad_fracturas,anio_carga,mes_carga
3179,68550,CASTILLO,2022-10-28,2022-10-29,2,2022,11
3297,68550,CASTILLO,2022-10-28,2022-10-29,2,2023,2
3474,68651,CASTILLO,2023-06-29,2023-06-29,1,2023,7
3516,68651,CASTILLO,2023-06-29,2023-06-29,1,2023,9
2036,161308,VACA MUERTA,2019-08-13,2019-09-08,30,2019,11
2037,161308,VACA MUERTA,2019-08-13,2019-09-08,30,2019,11
2790,162599,VACA MUERTA,2021-07-05,2021-07-22,47,2021,9
3275,162599,VACA MUERTA,2021-07-05,2021-07-22,47,2023,1
2791,162600,VACA MUERTA,2021-07-13,2021-07-22,26,2021,9
3276,162600,VACA MUERTA,2021-07-13,2021-07-22,26,2023,1


Registros eliminados: 6
Registros de fracturación restantes: 4884


#### 2.6. Fracturación: fechas y valores inválidos

En la sección 5.3 de la notebook 01 encontramos:

- fracturas con la fecha de fin anterior a la de inicio;
- fracturas con fecha de inicio posterior a la última carga de datos;
- presiones máximas iguales a 0.

Para incorporar la fracturación a cada mes de producción necesitamos una única fecha por evento, que indique a partir de cuándo esa información estaba disponible. Usamos la **fecha de fin de la fractura**, ya que recién cuando termina la operación se conocen la cantidad de fracturas y los volúmenes utilizados.

Cuando la fecha de fin es anterior a la de inicio no sabemos cuál de las dos es la correcta. En esos casos usamos la más tardía de las dos, que es el criterio más conservador: el evento se incorpora como mucho más tarde de lo que corresponde, pero nunca antes. Con las fechas futuras pasa lo mismo: quedan con una fecha posterior a todos los meses de producción, por lo que no se asocian a ningún registro.

Por último, una presión máxima igual a 0 no es posible durante una fractura, por lo que la tratamos como un valor no informado. Las presiones y potencias muy altas que identificamos en la notebook 01 no se modifican: pueden ser errores, pero no tenemos un criterio claro para distinguirlas de valores reales.

In [16]:
fecha_invertida = fracturacion["fecha_fin_fractura"] < fracturacion["fecha_inicio_fractura"]
print("Fracturas con fecha de fin anterior a la de inicio:", fecha_invertida.sum())

fracturacion["fecha_evento"] = fracturacion[["fecha_inicio_fractura", "fecha_fin_fractura"]].max(axis=1)

ultimo_mes_produccion = produccion["fecha_data"].max()
print("Fracturas con fecha de evento posterior al último mes de producción:", (fracturacion["fecha_evento"] > ultimo_mes_produccion).sum())

presion_cero = fracturacion["presion_maxima_psi"] == 0
print("Fracturas con presión máxima igual a 0:", presion_cero.sum())
fracturacion.loc[presion_cero, "presion_maxima_psi"] = np.nan

Fracturas con fecha de fin anterior a la de inicio: 41
Fracturas con fecha de evento posterior al último mes de producción: 3
Fracturas con presión máxima igual a 0: 78


La fecha del evento indica cuándo ocurrió la fractura, pero no cuándo esa información estuvo disponible en el dataset. Para dimensionar esa diferencia, calculamos cuántos meses pasaron entre el fin de cada fractura y su carga (`anio_carga` y `mes_carga`).

In [17]:
demora_carga = (
    (fracturacion["anio_carga"] - fracturacion["fecha_evento"].dt.year) * 12
    + (fracturacion["mes_carga"] - fracturacion["fecha_evento"].dt.month)
)

print("Meses entre el evento y su carga:")
display(demora_carga.describe())

Meses entre el evento y su carga:


count    4884.000000
mean       28.292179
std        32.028486
min        -6.000000
25%         3.000000
50%        13.000000
75%        49.000000
max       170.000000
dtype: float64

La mitad de las fracturas se cargó con más de un año de demora. Esto se debe en gran parte a que en 2019 se cargó de una sola vez la información histórica. Tenemos en cuenta esta diferencia en las limitaciones (sección 5.5).

#### 2.7. Variables categóricas

Verificamos que las variables categóricas de producción no tengan valores escritos de distintas formas (por ejemplo, con mayúsculas y minúsculas o con espacios al final). Las formaciones ya fueron unificadas en la sección 4.6 de la notebook 01.

In [18]:
columnas_categoricas = [
    "tipoestado", "tipoextraccion", "tipopozo", "empresa", "cuenca", "provincia",
    "areapermisoconcesion", "areayacimiento", "formacion", "sub_tipo_recurso",
    "clasificacion", "subclasificacion", "proyecto",
]

for columna in columnas_categoricas:
    valores = produccion[columna].dropna()
    normalizados = valores.str.strip().str.upper()
    print(f"{columna}: {valores.nunique()} categorías | {normalizados.nunique()} al normalizar | faltantes: {produccion[columna].isna().sum()}")

tipoestado: 16 categorías | 16 al normalizar | faltantes: 614


tipoextraccion: 11 categorías | 11 al normalizar | faltantes: 614


tipopozo: 7 categorías | 7 al normalizar | faltantes: 614


empresa: 55 categorías | 55 al normalizar | faltantes: 0


cuenca: 4 categorías | 4 al normalizar | faltantes: 0


...

formacion: 29 categorías | 29 al normalizar | faltantes: 0


sub_tipo_recurso: 2 categorías | 2 al normalizar | faltantes: 452


clasificacion: 3 categorías | 3 al normalizar | faltantes: 922


subclasificacion: 8 categorías | 8 al normalizar | faltantes: 922


proyecto: 2 categorías | 2 al normalizar | faltantes: 0


La cantidad de categorías es la misma antes y después de normalizar, por lo que no hay valores escritos de distintas formas.

Las variables de estado (`tipoestado`, `tipoextraccion` y `tipopozo`) tienen unos pocos registros sin informar (0,15%). Los dejamos como faltantes, ya que no hay forma confiable de completarlos.

#### 2.8. Variables que cambian a lo largo del tiempo

Antes de integrar los datos, verificamos qué atributos de producción son fijos para cada pozo y cuáles cambian de un mes a otro. Esto es importante porque un atributo que cambia en el tiempo debe tomarse del mes correspondiente, y no de una foto del estado actual del pozo.

In [19]:
atributos = [
    "tipoestado", "tipoextraccion", "tipopozo", "empresa", "formacion", "profundidad",
    "coordenadax", "coordenaday", "cuenca", "areayacimiento", "sub_tipo_recurso", "proyecto",
]

porcentaje_variable = (
    produccion.groupby("idpozo")[atributos].nunique().gt(1).mean() * 100
).round(1)

print("Porcentaje de pozos cuyo valor cambia a lo largo del tiempo:")
display(porcentaje_variable.sort_values(ascending=False))

Porcentaje de pozos cuyo valor cambia a lo largo del tiempo:


tipoestado          56.7
tipoextraccion      47.8
empresa             19.8
tipopozo            18.4
formacion            0.0
profundidad          0.0
coordenadax          0.0
coordenaday          0.0
cuenca               0.0
areayacimiento       0.0
sub_tipo_recurso     0.0
proyecto             0.0
dtype: float64

El estado del pozo, el sistema de extracción, el tipo de pozo y la empresa operadora cambian a lo largo del tiempo en una parte importante de los pozos. En cambio, la ubicación, la formación, la profundidad y el resto de las características geológicas son fijas.

Por lo tanto, en la tabla final conservamos los valores mensuales del dataset de producción para las variables que cambian en el tiempo.

### 3. Integración de los datasets

#### 3.1. Granularidad y estrategia de integración

En la sección 2.8 de la notebook 01 identificamos la granularidad de cada dataset:

| Dataset | Granularidad | Relación con producción |
|---|---|---|
| Producción | Una fila por pozo y mes | Tabla principal |
| Pozos | Una fila por pozo | Muchos a uno |
| Fracturación | Uno o más eventos por pozo | Muchos a muchos, si se combinara directamente |

A partir de esto definimos cómo incorporar cada fuente:

- **Producción** es la tabla principal y define la granularidad final: una fila por pozo y mes.
- **Pozos** se combina por `idpozo`. Como cada pozo tiene una sola fila, la combinación no puede multiplicar registros.
- **Fracturación** no puede combinarse directamente por `idpozo`, porque los pozos con más de un evento duplicarían sus filas de producción. Además, si la fracturación cambia con el tiempo, asignar todos los eventos del pozo a todos sus meses incorporaría información del futuro.

Verificamos esto último comparando la fecha de cada fractura con el primer mes de producción del pozo.

In [20]:
primer_mes_produccion = produccion.groupby("idpozo")["fecha_data"].min().dt.to_period("M")
mes_fractura = fracturacion["fecha_evento"].dt.to_period("M")
mes_inicio = fracturacion["idpozo"].map(primer_mes_produccion)

fracturas_con_produccion = mes_inicio.notna()
posterior_al_inicio = mes_fractura[fracturas_con_produccion] > mes_inicio[fracturas_con_produccion]

print("Pozos con más de un evento de fracturación:", (fracturacion.groupby("idpozo").size() > 1).sum())
print("Eventos de fracturación de pozos con producción:", fracturas_con_produccion.sum())
print(f"Eventos posteriores al primer mes de producción del pozo: {posterior_al_inicio.sum()} ({posterior_al_inicio.mean() * 100:.1f}%)")

Pozos con más de un evento de fracturación: 177
Eventos de fracturación de pozos con producción: 3903
Eventos posteriores al primer mes de producción del pozo: 559 (14.3%)


Una parte de las fracturas ocurre después de que el pozo comenzó a producir, y hay pozos con varios eventos en fechas distintas. Por eso agregamos la fracturación para cada pozo y mes, considerando solo los eventos ocurridos hasta ese mes.

#### 3.2. Preparación de la tabla principal

Creamos la columna `fecha`, que representa el mes de cada registro con el primer día del mes, y ordenamos los registros por pozo y fecha.

In [21]:
produccion["fecha"] = produccion["fecha_data"].dt.to_period("M").dt.to_timestamp()

produccion = produccion.sort_values(["idpozo", "fecha"]).reset_index(drop=True)

display(produccion[["idpozo", "anio", "mes", "fecha_data", "fecha"]].head())

,idpozo,anio,mes,fecha_data,fecha
0,3640,2006,2,2006-02-28,2006-02-01
1,3640,2006,3,2006-03-31,2006-03-01
2,3640,2006,4,2006-04-30,2006-04-01
3,3640,2006,5,2006-05-31,2006-05-01
4,3640,2006,6,2006-06-30,2006-06-01


#### 3.3. Selección de columnas del dataset de pozos

Antes de combinar, comparamos los atributos que el dataset de pozos comparte con el de producción.

In [22]:
atributos_pozo = (
    produccion
    .groupby("idpozo")[["sigla", "formacion", "cuenca", "profundidad"]]
    .first()
)

comparacion_pozos = atributos_pozo.join(
    pozos.set_index("idpozo")[["sigla", "formacion", "cuenca", "profundidad"]],
    rsuffix="_pozos",
    how="inner",
)

for columna in ["sigla", "formacion", "cuenca"]:
    iguales = (comparacion_pozos[columna] == comparacion_pozos[f"{columna}_pozos"]).mean() * 100
    print(f"{columna}: {iguales:.1f}% de coincidencia")

profundidad_valida = comparacion_pozos["profundidad"].notna()
iguales = np.isclose(
    comparacion_pozos.loc[profundidad_valida, "profundidad"],
    comparacion_pozos.loc[profundidad_valida, "profundidad_pozos"],
).mean() * 100
print(f"profundidad: {iguales:.1f}% de coincidencia (entre las profundidades válidas)")

sigla: 100.0% de coincidencia
formacion: 100.0% de coincidencia
cuenca: 100.0% de coincidencia
profundidad: 100.0% de coincidencia (entre las profundidades válidas)


La sigla, la formación, la cuenca y la profundidad coinciden en todos los pozos, por lo que no es necesario incorporarlas desde el dataset de pozos.

Revisamos el resto de las columnas de este dataset para decidir cuáles aportan información nueva.

In [23]:
pozos_con_produccion = pozos[pozos["idpozo"].isin(pozos_produccion)].set_index("idpozo")

primer_mes = produccion.groupby("idpozo")["fecha"].min()
fecha_fin_terminacion = pozos_con_produccion["adjiv_fecha_fin_term"].reindex(primer_mes.index)
terminacion_valida = fecha_fin_terminacion.dt.year > 1900

print("Pozos con fecha de fin de terminación faltante o del año 1900:", (~terminacion_valida).sum())
print(
    "Pozos con fecha de fin de terminación posterior a su primer mes de producción:",
    (fecha_fin_terminacion[terminacion_valida] > primer_mes[terminacion_valida] + pd.offsets.MonthEnd(0)).sum(),
)

proyecto_gas_plus = produccion.groupby("idpozo")["proyecto"].first().eq("GAS PLUS")
coincide_gasplus = (pozos_con_produccion["gasplus"].reindex(proyecto_gas_plus.index) == proyecto_gas_plus).mean() * 100
print(f"Coincidencia entre gasplus (pozos) y proyecto GAS PLUS (producción): {coincide_gasplus:.1f}%")

ultimo_tipopozo = produccion.groupby("idpozo")["tipopozo"].last()
coincide_tipopozo = (pozos_con_produccion["tipopozo"].reindex(ultimo_tipopozo.index) == ultimo_tipopozo).mean() * 100
print(f"Coincidencia entre tipopozo (pozos) y el tipo de pozo del último mes de producción: {coincide_tipopozo:.1f}%")

print("Cota de los pozos con producción:")
display(pozos_con_produccion["cota"].describe())

Pozos con fecha de fin de terminación faltante o del año 1900: 214
Pozos con fecha de fin de terminación posterior a su primer mes de producción: 819
Coincidencia entre gasplus (pozos) y proyecto GAS PLUS (producción): 98.4%
Coincidencia entre tipopozo (pozos) y el tipo de pozo del último mes de producción: 97.9%
Cota de los pozos con producción:


count    5006.000000
mean      492.040088
std       189.851293
min         0.000000
25%       373.895000
50%       471.730000
75%       543.892500
max      2770.000000
Name: cota, dtype: float64

A partir de esta revisión decidimos qué columnas del dataset de pozos incorporar:

| Columnas | Decisión | Motivo |
|---|---|---|
| `cota` | Se incorpora | Altura del terreno en la boca del pozo. Es la única característica fija que no está en el dataset de producción. |
| `sigla`, `formacion`, `cuenca`, `profundidad`, `area`, `yacimiento`, `provincia`, `empresa`, `clasificacion`, `subclasificacion`, `sub_tipo_recurso`, `tipo_recurso` | No se incorporan | Repiten información del dataset de producción. |
| `tipoestado`, `tipopozo`, `tipoextraccion` | No se incorporan | Representan el estado actual del pozo. Usarlos en meses anteriores sería incorporar información del futuro (fuga de información). Usamos los valores mensuales de producción. |
| `gasplus` | No se incorpora | Repite la información de `proyecto` del dataset de producción, que además es mensual. |
| Fechas de perforación y terminación | No se incorporan | Muchas son del año 1900 o posteriores al primer mes de producción del pozo, por lo que no son confiables. La antigüedad del pozo se calcula a partir de su historial de producción (sección 4). |
| `geojson`, `geom` | No se incorporan | Repiten las coordenadas que ya están en producción. |

#### 3.4. Combinación con el dataset de pozos

Combinamos con un `merge` de tipo `left`, para conservar todos los registros de producción aunque el pozo no figure en el dataset de pozos. Con `validate="many_to_one"` pandas verifica que cada pozo aparezca una sola vez en el dataset de pozos; si no fuera así, la combinación fallaría en lugar de multiplicar filas.

In [24]:
filas_antes = len(produccion)

obt = produccion.merge(
    pozos[["idpozo", "cota"]],
    on="idpozo",
    how="left",
    validate="many_to_one",
)

print("Filas antes de la combinación:", filas_antes)
print("Filas después de la combinación:", len(obt))
print("Registros sin información del dataset de pozos:", obt["cota"].isna().sum())
print("Pozos sin información del dataset de pozos:", obt.loc[obt["cota"].isna(), "idpozo"].nunique())

Filas antes de la combinación: 421046
Filas después de la combinación: 421046
Registros sin información del dataset de pozos: 118
Pozos sin información del dataset de pozos: 83


#### 3.5. Agregación de la fracturación por pozo y mes

Para cada pozo y mes queremos conocer la información de fracturación disponible hasta ese mes. Para lograrlo seguimos tres pasos:

1. Agrupamos los eventos de fracturación por pozo y mes del evento. Así, si un pozo tuvo dos fracturas en el mismo mes, quedan en una sola fila.
2. Para cada pozo, acumulamos los eventos en orden cronológico: la cantidad total de fracturas y los volúmenes inyectados se suman, mientras que la longitud de la rama, la presión y la potencia toman el valor máximo alcanzado hasta ese momento.
3. Asociamos a cada mes de producción la última fila acumulada cuya fecha sea igual o anterior a ese mes. Para esto usamos `pd.merge_asof`, que busca el valor más reciente sin mirar hacia adelante.

De esta manera, una fractura realizada en 2023 solo aparece en los registros de 2023 en adelante, y nunca en los meses anteriores.

In [25]:
fracturacion_pozos = fracturacion[fracturacion["idpozo"].isin(pozos_produccion)].copy()

fracturacion_pozos["mes_evento"] = fracturacion_pozos["fecha_evento"].dt.to_period("M").dt.to_timestamp()
fracturacion_pozos["arena_bombeada_tn"] = (
    fracturacion_pozos["arena_bombeada_nacional_tn"] + fracturacion_pozos["arena_bombeada_importada_tn"]
)

# Paso 1: un registro por pozo y mes del evento
fracturacion_mensual = (
    fracturacion_pozos
    .sort_values(["idpozo", "fecha_evento"])
    .groupby(["idpozo", "mes_evento"], as_index=False)
    .agg(
        eventos=("id_base_fractura_adjiv", "count"),
        fecha_evento=("fecha_evento", "max"),
        cantidad_fracturas=("cantidad_fracturas", "sum"),
        agua_inyectada_m3=("agua_inyectada_m3", "sum"),
        arena_bombeada_tn=("arena_bombeada_tn", "sum"),
        longitud_rama_horizontal_m=("longitud_rama_horizontal_m", "max"),
        presion_maxima_psi=("presion_maxima_psi", "max"),
        potencia_equipos_fractura_hp=("potencia_equipos_fractura_hp", "max"),
        tipo_terminacion=("tipo_terminacion", "last"),
    )
    .sort_values(["idpozo", "mes_evento"])
)

# Paso 2: acumulación cronológica dentro de cada pozo
por_pozo = fracturacion_mensual.groupby("idpozo")

fracturacion_acumulada = pd.DataFrame({
    "idpozo": fracturacion_mensual["idpozo"],
    "mes_evento": fracturacion_mensual["mes_evento"],
    "eventos_fractura_acum": por_pozo["eventos"].cumsum(),
    "cantidad_fracturas_acum": por_pozo["cantidad_fracturas"].cumsum(),
    "agua_inyectada_m3_acum": por_pozo["agua_inyectada_m3"].cumsum(),
    "arena_bombeada_tn_acum": por_pozo["arena_bombeada_tn"].cumsum(),
    "longitud_rama_horizontal_m_max": por_pozo["longitud_rama_horizontal_m"].cummax(),
    "presion_maxima_psi_max": por_pozo["presion_maxima_psi"].cummax(),
    "potencia_equipos_fractura_hp_max": por_pozo["potencia_equipos_fractura_hp"].cummax(),
    "tipo_terminacion_ultima": fracturacion_mensual["tipo_terminacion"],
    "fecha_primera_fractura": por_pozo["fecha_evento"].transform("min"),
    "fecha_ultima_fractura": fracturacion_mensual["fecha_evento"],
})

# cummax deja valores faltantes en los meses donde el dato no fue informado:
# completamos con el máximo alcanzado hasta el mes anterior del mismo pozo
columnas_maximo = ["presion_maxima_psi_max", "potencia_equipos_fractura_hp_max", "tipo_terminacion_ultima"]
fracturacion_acumulada[columnas_maximo] = fracturacion_acumulada.groupby("idpozo")[columnas_maximo].ffill()

print("Eventos de fracturación de pozos con producción:", len(fracturacion_pozos))
print("Filas por pozo y mes del evento:", len(fracturacion_mensual))
display(fracturacion_acumulada[fracturacion_acumulada["idpozo"].duplicated(keep=False)].head(6))

Eventos de fracturación de pozos con producción: 3903
Filas por pozo y mes del evento: 3870


,idpozo,mes_evento,eventos_fractura_acum,cantidad_fracturas_acum,agua_inyectada_m3_acum,arena_bombeada_tn_acum,longitud_rama_horizontal_m_max,presion_maxima_psi_max,potencia_equipos_fractura_hp_max,tipo_terminacion_ultima,fecha_primera_fractura,fecha_ultima_fractura
19,136630,2011-10-01,1,5,404.0,130.0,0.0,5000.0,2500.0,Punzado,2011-10-01,2011-10-01
20,136630,2014-07-01,2,7,599.4,196.0,0.0,5000.0,2500.0,Punzado,2011-10-01,2014-07-07
21,136630,2015-06-01,3,9,685.4,234.0,0.0,5700.0,2500.0,Punzado,2011-10-01,2015-06-01
22,136856,2011-11-01,4,4,1412.0,4205.0,74.0,7960.0,NaN,Tapón disparo,2011-11-30,2011-11-30
23,136856,2011-12-01,5,5,1824.0,5883.0,76.0,7960.0,NaN,Tapón disparo,2011-11-30,2011-12-01
72,145002,2022-05-01,1,2,3252.1,245.2,77.6,7489.0,10885.0,Punzado,2022-05-24,2022-05-24


La primera fecha de fractura se calcula sobre todos los eventos del pozo. Para que no incorpore información del futuro, más adelante solo la conservamos en los meses en los que ese primer evento ya ocurrió, lo que se cumple automáticamente: `merge_asof` solo asocia una fila acumulada a los meses posteriores a su primer evento.

Ahora asociamos la fracturación acumulada a cada mes de producción.

In [26]:
filas_antes = len(obt)

obt = pd.merge_asof(
    obt.sort_values("fecha"),
    fracturacion_acumulada.sort_values("mes_evento"),
    left_on="fecha",
    right_on="mes_evento",
    by="idpozo",
    direction="backward",
)

obt = obt.drop(columns="mes_evento").sort_values(["idpozo", "fecha"]).reset_index(drop=True)

obt["tiene_datos_fractura"] = obt["idpozo"].isin(fracturacion_pozos["idpozo"])
obt["fractura_previa"] = obt["eventos_fractura_acum"].notna()

obt["meses_desde_ultima_fractura"] = (
    (obt["fecha"].dt.year - obt["fecha_ultima_fractura"].dt.year) * 12
    + (obt["fecha"].dt.month - obt["fecha_ultima_fractura"].dt.month)
)

print("Filas antes de la combinación:", filas_antes)
print("Filas después de la combinación:", len(obt))
print("Registros de pozos con datos de fracturación:", obt["tiene_datos_fractura"].sum())
print("Registros con al menos una fractura previa o en el mismo mes:", obt["fractura_previa"].sum())

Filas antes de la combinación: 421046
Filas después de la combinación: 421046
Registros de pozos con datos de fracturación: 277593
Registros con al menos una fractura previa o en el mismo mes: 273016


La cantidad de filas se mantiene, por lo que la combinación no generó registros duplicados.

Distinguimos dos situaciones con columnas separadas:

- `tiene_datos_fractura` indica si el pozo figura en el dataset de fracturación;
- `fractura_previa` indica si, en ese mes, ya había ocurrido al menos una fractura informada.

Un pozo puede tener datos de fracturación y, aun así, no tener fracturas previas en sus primeros meses de producción, si la fractura se informó con una fecha posterior.

### 4. Validación temporal y estructural

#### 4.1. Variables temporales del pozo

Para que las siguientes etapas puedan construir variables a partir del historial de cada pozo, agregamos tres columnas:

- `meses_desde_inicio`: meses transcurridos desde el primer registro del pozo, que usamos como medida de la antigüedad productiva del pozo. Más abajo verificamos que el primer registro corresponda efectivamente al inicio de la producción.
- `meses_desde_registro_anterior`: meses transcurridos desde el registro anterior del mismo pozo. Si vale 1, el mes anterior existe; si es mayor, hay meses faltantes en la serie.
- `inicio_censurado`: indica si el pozo ya tenía registros en el primer mes del dataset (enero de 2006). Para estos pozos no sabemos cuándo comenzaron a producir, por lo que `meses_desde_inicio` subestima su antigüedad.

In [27]:
indice_mes = obt["fecha"].dt.year * 12 + obt["fecha"].dt.month

obt["meses_desde_inicio"] = indice_mes - indice_mes.groupby(obt["idpozo"]).transform("min")
obt["meses_desde_registro_anterior"] = indice_mes.groupby(obt["idpozo"]).diff()

primer_mes_dataset = obt["fecha"].min()
obt["inicio_censurado"] = obt.groupby("idpozo")["fecha"].transform("min") == primer_mes_dataset

print("Primer mes del dataset:", primer_mes_dataset.date())
print("Pozos con inicio censurado:", obt.loc[obt["inicio_censurado"], "idpozo"].nunique())

print("Distribución de meses desde el registro anterior:")
display(obt["meses_desde_registro_anterior"].value_counts().head(8))

print("Pozos con al menos un mes faltante en su serie:", obt.loc[obt["meses_desde_registro_anterior"] > 1, "idpozo"].nunique())

Primer mes del dataset: 2006-01-01
Pozos con inicio censurado: 195
Distribución de meses desde el registro anterior:


meses_desde_registro_anterior
1.0     415126
2.0        610
3.0         44
58.0        16
4.0         13
61.0        10
7.0          8
5.0          7
Name: count, dtype: int64

Pozos con al menos un mes faltante en su serie: 704


Para verificar que el primer registro de cada pozo corresponde al inicio de su producción, lo comparamos con el padrón oficial de pozos con fecha de primera producción, que forma parte de los archivos descargados.

En ese padrón el mes es igual a 1 en casi todos los pozos, por lo que en la práctica solo informa el año. Por eso comparamos solamente el año.

In [28]:
padron = pd.read_csv(ruta_raw / "padrn-de-pozos-de-captulo-iv-con-fecha-de-primera-produccin.csv")

print(f"Pozos del padrón con mes igual a 1: {(padron['mes'] == 1).mean() * 100:.1f}%")

primer_anio = obt.groupby("idpozo")["anio"].min()
anio_padron = padron.set_index("idpozo")["anio"].reindex(primer_anio.index)

con_padron = anio_padron.notna()
coincide_anio = primer_anio[con_padron] == anio_padron[con_padron]

print("Pozos con producción que figuran en el padrón:", con_padron.sum())
print(f"Pozos cuyo primer año de registro coincide con el padrón: {coincide_anio.mean() * 100:.1f}%")

Pozos del padrón con mes igual a 1: 99.6%
Pozos con producción que figuran en el padrón: 5044
Pozos cuyo primer año de registro coincide con el padrón: 100.0%


El año del primer registro coincide con el año de primera producción del padrón en casi todos los pozos. Por lo tanto, `meses_desde_inicio` es una buena aproximación de la antigüedad productiva, salvo en los pozos con inicio censurado.

La gran mayoría de los registros sigue inmediatamente al mes anterior del pozo, pero hay pozos con meses faltantes en su serie, en algunos casos de varios años.

No completamos esos meses con filas nuevas, ya que no sabemos si el pozo estuvo parado o si el registro no fue cargado, y agregarlos implicaría inventar observaciones. En cambio, la columna `meses_desde_registro_anterior` deja identificados los huecos para que la etapa de ingeniería de variables los tenga en cuenta: **los rezagos y promedios móviles deben calcularse por fecha y no por posición de la fila**, ya que de lo contrario se asociarían meses que no son consecutivos.

#### 4.2. Construcción de la variable objetivo

La pregunta del proyecto es cuánto petróleo va a producir un pozo el próximo mes. Por lo tanto, la variable objetivo de cada fila es `prod_pet_mes_siguiente`: la producción de petróleo del mismo pozo en el mes calendario siguiente.

Siguiendo el criterio anterior, no la calculamos con `shift(-1)`, que tomaría la fila siguiente aunque correspondiera a un mes posterior. En su lugar, buscamos explícitamente el registro del mes siguiente: si ese mes no existe, la variable objetivo queda como faltante.

In [29]:
mes_siguiente = obt[["idpozo", "fecha", "prod_pet"]].copy()
mes_siguiente["fecha"] = mes_siguiente["fecha"] - pd.DateOffset(months=1)
mes_siguiente = mes_siguiente.rename(columns={"prod_pet": "prod_pet_mes_siguiente"})

filas_antes = len(obt)

obt = obt.merge(mes_siguiente, on=["idpozo", "fecha"], how="left", validate="one_to_one")
obt["target_disponible"] = obt["prod_pet_mes_siguiente"].notna()

print("Filas antes de la combinación:", filas_antes)
print("Filas después de la combinación:", len(obt))
print("Registros con variable objetivo disponible:", obt["target_disponible"].sum())
print("Registros sin variable objetivo:", (~obt["target_disponible"]).sum())

Filas antes de la combinación: 421046
Filas después de la combinación: 421046
Registros con variable objetivo disponible: 415126
Registros sin variable objetivo: 5920


Los registros sin variable objetivo corresponden al último mes de cada pozo y a los meses que están seguidos por un hueco en la serie. Los conservamos porque su información es necesaria para calcular el historial de los meses siguientes; simplemente no pueden usarse como ejemplos de entrenamiento.

Verificamos que la variable objetivo esté bien construida: en los registros donde el mes siguiente existe, debe coincidir con la producción de la fila siguiente del mismo pozo.

In [30]:
siguiente_fila = obt.groupby("idpozo")["prod_pet"].shift(-1)
siguiente_es_consecutivo = obt.groupby("idpozo")["meses_desde_registro_anterior"].shift(-1) == 1

coincide_target = np.isclose(
    obt.loc[siguiente_es_consecutivo, "prod_pet_mes_siguiente"],
    siguiente_fila[siguiente_es_consecutivo],
)

print("Registros con mes siguiente consecutivo:", siguiente_es_consecutivo.sum())
print("Registros donde la variable objetivo coincide con la fila siguiente:", coincide_target.sum())
print("Registros con variable objetivo pero sin mes siguiente consecutivo:", (obt["target_disponible"] & ~siguiente_es_consecutivo).sum())

Registros con mes siguiente consecutivo: 415126
Registros donde la variable objetivo coincide con la fila siguiente: 415126
Registros con variable objetivo pero sin mes siguiente consecutivo: 0


#### 4.3. Período de modelado

En las conclusiones de la notebook 01 definimos que, para el entrenamiento y la proyección de los modelos, deberían usarse como máximo los registros de 2025 y 2026.

Sin embargo, conservamos toda la historia en la tabla final: para calcular, por ejemplo, la producción de los últimos meses de un registro de enero de 2025, se necesitan los registros de 2024. Por eso agregamos la columna `periodo_modelado`, que identifica los registros de 2025 en adelante, y dejamos el filtro para la etapa de modelado.

In [31]:
obt["periodo_modelado"] = obt["anio"] >= 2025

print("Registros del período de modelado:", obt["periodo_modelado"].sum())
print("Registros del período de modelado con variable objetivo:", (obt["periodo_modelado"] & obt["target_disponible"]).sum())
print("Pozos con registros en el período de modelado:", obt.loc[obt["periodo_modelado"], "idpozo"].nunique())

Registros del período de modelado: 88610
Registros del período de modelado con variable objetivo: 83509
Pozos con registros en el período de modelado: 5071


#### 4.4. Selección y orden de las columnas

Definimos las columnas de la tabla final, agrupadas según su función. Las columnas de producción que no incorporamos, y el motivo, se detallan en la sección 5.4.

In [32]:
columnas_identificacion = ["idpozo", "sigla"]

columnas_temporales = [
    "fecha", "anio", "mes", "meses_desde_inicio", "meses_desde_registro_anterior",
    "inicio_censurado", "periodo_modelado",
]

columnas_produccion = [
    "prod_pet", "prod_gas", "prod_agua", "iny_agua", "iny_gas", "tef",
]

columnas_estado = ["tipoestado", "tipoextraccion", "tipopozo", "empresa"]

columnas_pozo = [
    "cuenca", "provincia", "areapermisoconcesion", "areayacimiento", "formacion",
    "sub_tipo_recurso", "clasificacion", "subclasificacion", "proyecto",
    "profundidad", "cota", "coordenadax", "coordenaday",
]

columnas_fractura = [
    "tiene_datos_fractura", "fractura_previa", "eventos_fractura_acum",
    "cantidad_fracturas_acum", "agua_inyectada_m3_acum", "arena_bombeada_tn_acum",
    "longitud_rama_horizontal_m_max", "presion_maxima_psi_max",
    "potencia_equipos_fractura_hp_max", "tipo_terminacion_ultima",
    "fecha_primera_fractura", "fecha_ultima_fractura", "meses_desde_ultima_fractura",
]

columnas_control = ["prod_corregida", "tef_corregido", "target_disponible"]

columnas_objetivo = ["prod_pet_mes_siguiente"]

columnas_finales = (
    columnas_identificacion + columnas_temporales + columnas_produccion + columnas_estado
    + columnas_pozo + columnas_fractura + columnas_control + columnas_objetivo
)

dataset_modelado = obt[columnas_finales].copy()

print("Columnas de la tabla final:", dataset_modelado.shape[1])
print("Columnas sin asignar a ningún grupo:", sorted(set(obt.columns) - set(columnas_finales)))

Columnas de la tabla final: 49
Columnas sin asignar a ningún grupo: ['fecha_data', 'fechaingreso', 'formprod', 'habilitado', 'idareapermisoconcesion', 'idareayacimiento', 'idempresa', 'idusuario', 'iny_co2', 'iny_otro', 'observaciones', 'rectificado', 'tipo_de_recurso', 'vida_util']


#### 4.5. Validaciones finales

Antes de exportar, verificamos explícitamente las condiciones que debe cumplir la tabla. Cada validación usa `assert`: si alguna no se cumple, la ejecución se detiene en lugar de exportar un dataset con errores.

In [33]:
validaciones = {}

# Granularidad y cantidad de filas
validaciones["Una sola fila por pozo y mes"] = not dataset_modelado.duplicated(["idpozo", "fecha"]).any()
validaciones["Misma cantidad de filas que producción"] = len(dataset_modelado) == filas_iniciales["produccion"]
validaciones["Mismos pozos que producción"] = dataset_modelado["idpozo"].nunique() == len(pozos_produccion)

# Orden cronológico
validaciones["Ordenado por pozo y fecha"] = dataset_modelado.equals(
    dataset_modelado.sort_values(["idpozo", "fecha"])
)
validaciones["Meses consecutivos o con hueco identificado"] = (
    dataset_modelado["meses_desde_registro_anterior"].dropna() >= 1
).all()

# Valores de producción
validaciones["Sin producciones negativas"] = (dataset_modelado[columnas_produccion] < 0).sum().sum() == 0
validaciones["tef dentro de los días del mes"] = (
    dataset_modelado["tef"].dropna() <= dataset_modelado.loc[dataset_modelado["tef"].notna(), "fecha"].dt.days_in_month
).all()
validaciones["Variable objetivo sin valores negativos"] = (dataset_modelado["prod_pet_mes_siguiente"].dropna() >= 0).all()

# Fuga de información
validaciones["Fracturas asociadas solo a meses posteriores o iguales"] = (
    dataset_modelado["fecha_ultima_fractura"].dropna()
    <= dataset_modelado.loc[dataset_modelado["fecha_ultima_fractura"].notna(), "fecha"] + pd.offsets.MonthEnd(0)
).all()
validaciones["Primera fractura anterior o igual al mes del registro"] = (
    dataset_modelado["fecha_primera_fractura"].dropna()
    <= dataset_modelado.loc[dataset_modelado["fecha_primera_fractura"].notna(), "fecha"] + pd.offsets.MonthEnd(0)
).all()
validaciones["Meses desde la última fractura no negativos"] = (dataset_modelado["meses_desde_ultima_fractura"].dropna() >= 0).all()
validaciones["Variable objetivo solo con mes siguiente consecutivo"] = (
    (dataset_modelado["target_disponible"] & ~siguiente_es_consecutivo).sum() == 0
)

resultado_validaciones = pd.Series(validaciones, name="cumple")
display(resultado_validaciones)

assert resultado_validaciones.all(), "Hay validaciones que no se cumplen"

Una sola fila por pozo y mes                              True
Misma cantidad de filas que producción                    True
Mismos pozos que producción                               True
Ordenado por pozo y fecha                                 True
Meses consecutivos o con hueco identificado               True
Sin producciones negativas                                True
tef dentro de los días del mes                            True
Variable objetivo sin valores negativos                   True
Fracturas asociadas solo a meses posteriores o iguales    True
Primera fractura anterior o igual al mes del registro     True
Meses desde la última fractura no negativos               True
Variable objetivo solo con mes siguiente consecutivo      True
Name: cumple, dtype: bool

Verificamos también los tipos de datos de la tabla final.

In [34]:
display(dataset_modelado.dtypes.value_counts())
display(dataset_modelado.dtypes.to_frame("tipo").T)

float64           20
str               15
bool               7
int64              3
datetime64[us]     3
int32              1
Name: count, dtype: int64

,idpozo,sigla,fecha,anio,mes,meses_desde_inicio,meses_desde_registro_anterior,inicio_censurado,periodo_modelado,prod_pet,prod_gas,prod_agua,iny_agua,iny_gas,tef,tipoestado,tipoextraccion,tipopozo,empresa,cuenca,provincia,areapermisoconcesion,areayacimiento,formacion,sub_tipo_recurso,clasificacion,subclasificacion,proyecto,profundidad,cota,coordenadax,coordenaday,tiene_datos_fractura,fractura_previa,eventos_fractura_acum,cantidad_fracturas_acum,agua_inyectada_m3_acum,arena_bombeada_tn_acum,longitud_rama_horizontal_m_max,presion_maxima_psi_max,potencia_equipos_fractura_hp_max,tipo_terminacion_ultima,fecha_primera_fractura,fecha_ultima_fractura,meses_desde_ultima_fractura,prod_corregida,tef_corregido,target_disponible,prod_pet_mes_siguiente
tipo,int64,str,datetime64[us],int64,int64,int32,float64,bool,bool,float64,float64,float64,float64,float64,float64,str,str,str,str,str,str,str,str,str,str,str,str,str,float64,float64,float64,float64,bool,bool,float64,float64,float64,float64,float64,float64,float64,str,datetime64[us],datetime64[us],float64,bool,bool,bool,float64


#### 4.6. Distribución de los valores faltantes

Por último, revisamos qué columnas tienen valores faltantes y en qué proporción, tanto en toda la tabla como en el período de modelado.

In [35]:
faltantes = pd.DataFrame({
    "faltantes": dataset_modelado.isna().sum(),
    "porcentaje": (dataset_modelado.isna().mean() * 100).round(2),
    "porcentaje_periodo_modelado": (dataset_modelado[dataset_modelado["periodo_modelado"]].isna().mean() * 100).round(2),
})

display(faltantes[faltantes["faltantes"] > 0].sort_values("porcentaje", ascending=False))

,faltantes,porcentaje,porcentaje_periodo_modelado
potencia_equipos_fractura_hp_max,150447,35.73,24.60
presion_maxima_psi_max,148133,35.18,24.32
cantidad_fracturas_acum,148030,35.16,24.30
agua_inyectada_m3_acum,148030,35.16,24.30
meses_desde_ultima_fractura,148030,35.16,24.30
fecha_ultima_fractura,148030,35.16,24.30
fecha_primera_fractura,148030,35.16,24.30
tipo_terminacion_ultima,148030,35.16,24.30
longitud_rama_horizontal_m_max,148030,35.16,24.30
arena_bombeada_tn_acum,148030,35.16,24.30


Los valores faltantes tienen explicaciones concretas:

- las variables de fracturación faltan en los pozos sin datos de fracturación y en los meses anteriores a la primera fractura informada;
- la variable objetivo falta en el último mes de cada pozo y antes de cada hueco de la serie;
- `meses_desde_registro_anterior` falta en el primer registro de cada pozo;
- `profundidad` falta en los pozos cuya profundidad era 0 o claramente errónea;
- `cota` falta en los pozos que no figuran en el dataset de pozos;
- `tef` falta en los registros con valores irrecuperables, y las variables de estado y de clasificación faltan en los pocos registros sin informar.

En ningún caso completamos estos valores, ya que hacerlo implicaría inventar información. La decisión sobre cómo tratarlos depende del modelo, por lo que la dejamos para la etapa de modelado.

### 5. Exportación del dataset de modelado

#### 5.1. Exportación

Guardamos la tabla final en `data/processed/dataset_modelado.csv`. Las fechas se guardan en formato `año-mes-día`, por lo que al leer el archivo hay que volver a convertirlas con `parse_dates`.

In [36]:
ruta_salida = ruta_processed / "dataset_modelado.csv"

dataset_modelado.to_csv(ruta_salida, index=False, date_format="%Y-%m-%d")

print(f"Guardado: {ruta_salida} | Filas: {len(dataset_modelado):,} | Columnas: {dataset_modelado.shape[1]}")

Guardado: ../data/processed/dataset_modelado.csv | Filas: 421,046 | Columnas: 49


Para verificar que la exportación sea correcta, leemos el archivo y lo comparamos con la tabla en memoria.

In [37]:
columnas_fecha = ["fecha", "fecha_primera_fractura", "fecha_ultima_fractura"]
dataset_leido = pd.read_csv(ruta_salida, parse_dates=columnas_fecha, low_memory=False)

print("Misma cantidad de filas y columnas:", dataset_leido.shape == dataset_modelado.shape)
print("Mismas columnas:", list(dataset_leido.columns) == list(dataset_modelado.columns))
print("Mismas combinaciones de pozo y fecha:", dataset_leido[["idpozo", "fecha"]].equals(dataset_modelado[["idpozo", "fecha"]]))
print(
    "Misma variable objetivo:",
    np.allclose(dataset_leido["prod_pet_mes_siguiente"], dataset_modelado["prod_pet_mes_siguiente"], equal_nan=True),
)

Misma cantidad de filas y columnas: True
Mismas columnas: True
Mismas combinaciones de pozo y fecha: True
Misma variable objetivo: True


#### 5.2. Resumen de calidad

Comparamos los datasets de entrada con la tabla final.

In [38]:
resumen_calidad = pd.DataFrame({
    **resumen_entrada,
    "Dataset de modelado": resumen(dataset_modelado, "fecha"),
    "Dataset de modelado (2025 y 2026)": resumen(dataset_modelado[dataset_modelado["periodo_modelado"]], "fecha"),
}).T

display(resumen_calidad)

print("Registros con variable objetivo:", dataset_modelado["target_disponible"].sum())
print("Registros con variable objetivo en el período de modelado:", (dataset_modelado["periodo_modelado"] & dataset_modelado["target_disponible"]).sum())
print("Registros con producción corregida:", dataset_modelado["prod_corregida"].sum())
print("Registros con tef corregido:", dataset_modelado["tef_corregido"].sum())
print("Registros con tef corregido en el período de modelado:", (dataset_modelado["periodo_modelado"] & dataset_modelado["tef_corregido"]).sum())

,filas,pozos,columnas,desde,hasta
Producción (entrada),421046,5089,40,2006-01,2026-07
Pozos (entrada),85611,85611,26,-,-
Fracturación (entrada),4890,4646,30,2006-05,2026-07
Dataset de modelado,421046,5089,49,2006-01,2026-07
Dataset de modelado (2025 y 2026),88610,5071,49,2025-01,2026-07


Registros con variable objetivo: 415126
Registros con variable objetivo en el período de modelado: 83509
Registros con producción corregida: 3
Registros con tef corregido: 71
Registros con tef corregido en el período de modelado: 34


#### 5.3. Diccionario de datos

Las unidades de producción fueron verificadas en la sección 1.5. Las de fracturación, profundidad y cota corresponden a las indicadas en el nombre o la descripción de cada variable en la fuente.

**Rol** indica cómo puede usarse cada columna en la etapa de modelado:

- *identificador*: identifica la fila, no debe usarse como variable del modelo;
- *temporal*: ubica el registro en el tiempo;
- *variable*: información disponible al cierre del mes de la fila, utilizable como variable del modelo;
- *control*: indica correcciones o disponibilidad de datos;
- *objetivo*: lo que se quiere predecir; nunca debe usarse como variable del modelo.

| Columna | Fuente | Descripción | Unidad | Rol |
|---|---|---|---|---|
| `idpozo` | Producción | Identificador del pozo. | - | identificador |
| `sigla` | Producción | Nombre o sigla del pozo. | - | identificador |
| `fecha` | Producción | Mes del registro, representado por el primer día del mes. | fecha | temporal |
| `anio` | Producción | Año del registro. | - | temporal |
| `mes` | Producción | Mes del registro (1 a 12). | - | temporal |
| `meses_desde_inicio` | Calculada | Meses desde el primer registro del pozo (antigüedad productiva). | meses | variable |
| `meses_desde_registro_anterior` | Calculada | Meses desde el registro anterior del mismo pozo. Mayor a 1 indica meses faltantes. | meses | temporal |
| `inicio_censurado` | Calculada | El pozo ya tenía registros en enero de 2006, por lo que su antigüedad real es mayor a `meses_desde_inicio`. | booleano | control |
| `periodo_modelado` | Calculada | El registro pertenece a 2025 o 2026, el período propuesto para entrenar los modelos. | booleano | control |
| `prod_pet` | Producción | Producción de petróleo del mes. Se corrigió un valor negativo. | m³ | variable |
| `prod_gas` | Producción | Producción de gas del mes. Se corrigieron dos valores negativos. | miles de m³ | variable |
| `prod_agua` | Producción | Producción de agua del mes. | m³ | variable |
| `iny_agua` | Producción | Agua inyectada en el mes. | m³ | variable |
| `iny_gas` | Producción | Gas inyectado en el mes. | miles de m³ | variable |
| `tef` | Producción | Días efectivos de producción del mes. Truncado a los días del mes o faltante en los valores imposibles. | días | variable |
| `tipoestado` | Producción | Estado del pozo informado en el mes. | - | variable |
| `tipoextraccion` | Producción | Sistema de extracción informado en el mes. | - | variable |
| `tipopozo` | Producción | Tipo de pozo informado en el mes (petrolífero, gasífero, etc.). | - | variable |
| `empresa` | Producción | Empresa operadora en el mes. | - | variable |
| `cuenca` | Producción | Cuenca del pozo. | - | variable |
| `provincia` | Producción | Provincia del pozo. | - | variable |
| `areapermisoconcesion` | Producción | Área de concesión. | - | variable |
| `areayacimiento` | Producción | Yacimiento. | - | variable |
| `formacion` | Producción | Formación geológica productiva (en mayúsculas). | - | variable |
| `sub_tipo_recurso` | Producción | Subtipo de recurso no convencional (SHALE o TIGHT). | - | variable |
| `clasificacion` | Producción | Clasificación del pozo (explotación, exploración o servicio). | - | variable |
| `subclasificacion` | Producción | Subclasificación del pozo (desarrollo, avanzada, etc.). | - | variable |
| `proyecto` | Producción | Proyecto al que pertenece el pozo (GAS PLUS o Sin Proyecto). | - | variable |
| `profundidad` | Producción | Profundidad del pozo. Faltante si era 0 o mayor a 10.000 m. | m | variable |
| `cota` | Pozos | Altura del terreno en la boca del pozo. | m | variable |
| `coordenadax` | Producción | Longitud. Se corrigió un pozo con coordenadas invertidas. | grados decimales | variable |
| `coordenaday` | Producción | Latitud. Se corrigió un pozo con coordenadas invertidas. | grados decimales | variable |
| `tiene_datos_fractura` | Calculada | El pozo figura en el dataset de fracturación. | booleano | control |
| `fractura_previa` | Calculada | Hay al menos una fractura informada hasta el mes del registro. | booleano | variable |
| `eventos_fractura_acum` | Fracturación | Eventos de fracturación informados hasta el mes del registro. | eventos | variable |
| `cantidad_fracturas_acum` | Fracturación | Cantidad de fracturas (etapas) acumuladas hasta el mes del registro. | fracturas | variable |
| `agua_inyectada_m3_acum` | Fracturación | Agua inyectada en fracturas hasta el mes del registro. | m³ | variable |
| `arena_bombeada_tn_acum` | Fracturación | Arena bombeada (nacional e importada) hasta el mes del registro. | toneladas | variable |
| `longitud_rama_horizontal_m_max` | Fracturación | Mayor longitud de rama horizontal informada hasta el mes del registro. | m | variable |
| `presion_maxima_psi_max` | Fracturación | Mayor presión máxima informada hasta el mes del registro. | psi | variable |
| `potencia_equipos_fractura_hp_max` | Fracturación | Mayor potencia de equipos informada hasta el mes del registro. | hp | variable |
| `tipo_terminacion_ultima` | Fracturación | Tipo de terminación del último evento de fracturación. | - | variable |
| `fecha_primera_fractura` | Fracturación | Fecha del primer evento de fracturación del pozo (solo en meses posteriores a ese evento). | fecha | variable |
| `fecha_ultima_fractura` | Fracturación | Fecha del último evento de fracturación hasta el mes del registro. | fecha | variable |
| `meses_desde_ultima_fractura` | Calculada | Meses entre la última fractura y el mes del registro. | meses | variable |
| `prod_corregida` | Calculada | Se corrigió `prod_pet` o `prod_gas` por tener un valor negativo. | booleano | control |
| `tef_corregido` | Calculada | Se truncó o se reemplazó por faltante el valor de `tef`. | booleano | control |
| `target_disponible` | Calculada | Existe el registro del mes siguiente, por lo que la variable objetivo está disponible. | booleano | control |
| `prod_pet_mes_siguiente` | Calculada | Producción de petróleo del mismo pozo en el mes siguiente. | m³ | objetivo |

#### 5.4. Decisiones de limpieza e integración

**Decisiones de limpieza**

| Problema | Registros afectados | Tratamiento |
|---|---|---|
| Producción de gas negativa | 2 | Se invirtió el signo (sección 2.1). |
| Producción de petróleo de -0,001 | 1 | Se reemplazó por 0 (sección 2.1). |
| `tef` mayor a los días del mes, exceso de hasta 5 días | 66 | Se truncó a los días del mes (sección 2.2). |
| `tef` mayor a los días del mes, exceso de más de 5 días | 5 | Se reemplazó por faltante (sección 2.2). |
| Profundidad igual a 0 o mayor a 10.000 m | 5.860 | Se reemplazó por faltante (sección 2.3). |
| Coordenadas invertidas | 68 | Se intercambiaron (sección 2.4). |
| Eventos de fracturación cargados dos veces | 6 | Se eliminó la carga repetida (sección 2.5). |
| Fracturas con fecha de fin anterior a la de inicio | 41 | Se usó la fecha más tardía (sección 2.6). |
| Presión máxima de fractura igual a 0 | 78 | Se reemplazó por faltante (sección 2.6). |
| Valores extremos de producción y producciones iguales a 0 | - | No se modificaron, ya que pueden corresponder a situaciones reales (sección 7.5 de la notebook 01). |

No se eliminó ningún registro de producción.

**Columnas de producción no incorporadas**

| Columnas | Motivo |
|---|---|
| `iny_co2`, `iny_otro`, `habilitado`, `tipo_de_recurso` | Tienen un único valor en todo el dataset (sección 4.2 de la notebook 01). |
| `vida_util` | Faltante en el 98% de los registros. |
| `observaciones` | Texto libre, faltante en el 94% de los registros. |
| `fechaingreso`, `idusuario`, `rectificado` | Datos administrativos de la carga, no del pozo. |
| `fecha_data` | Reemplazada por `fecha`. |
| `idempresa`, `idareapermisoconcesion`, `idareayacimiento` | Códigos que repiten la información de las columnas con el nombre. |
| `formprod` | Código abreviado de `formacion`. |

**Decisiones de integración**

- Producción es la tabla principal y define la granularidad: una fila por pozo y mes.
- Del dataset de pozos solo se incorpora `cota`; el resto de sus columnas repite información de producción, representa el estado actual del pozo o no es confiable (sección 3.3).
- La fracturación se agregó por pozo y mes y se acumuló en orden cronológico. Cada mes solo incluye los eventos ocurridos hasta ese mes, identificados por su fecha de fin (sección 3.5).
- No se agregaron filas para los meses faltantes de cada pozo; los huecos quedan identificados en `meses_desde_registro_anterior`.
- La variable objetivo se construyó buscando el mes calendario siguiente, no la fila siguiente.
- Se conserva toda la historia desde 2006. El período de modelado (2025 y 2026) se identifica con `periodo_modelado`.

#### 5.5. Limitaciones

- **Fecha de disponibilidad de la fracturación.** Asociamos cada fractura al mes en que terminó la operación. Sin embargo, esta información se cargó con demora: en la mitad de los casos, más de un año después del evento (sección 2.6). Si en la etapa de modelado se quisiera simular estrictamente la información disponible en cada momento, habría que usar la fecha de carga (`anio_carga` y `mes_carga`) en lugar de la fecha de la fractura.
- **Pozos sin datos de fracturación.** Una cuarta parte de los pozos no figura en el dataset de fracturación. Sus variables de fracturación son faltantes, lo que no significa que no hayan sido fracturados.
- **Valores de `tef` truncados.** En los registros copiados del mes anterior asumimos que el pozo produjo todo el mes. El valor real es desconocido; los registros quedan identificados en `tef_corregido`.
- **Valores extremos sin corregir.** Algunas presiones y potencias de fracturación muy altas y un pozo con cota igual a 0 se conservaron sin cambios, por no tener un criterio claro para distinguirlos de valores reales.
- **Antigüedad de los pozos más antiguos.** Para los pozos con registros desde enero de 2006, `meses_desde_inicio` subestima su antigüedad. Están identificados en `inicio_censurado`.
- **Año 2026 parcial.** El dataset llega hasta julio de 2026, por lo que ese mes no tiene variable objetivo.
- **Uso de `tef` y de las variables de estado.** Estas variables describen el mes de la fila, por lo que son válidas para predecir el mes siguiente. En cambio, los valores del mes que se quiere predecir no se conocen de antemano y no forman parte de la tabla.